# Step 2: Fetch dispensed UKB prescriptions data

In [ ]:
import pyspark
import dxpy
import hail as hl

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

### Environment setup check

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Script configuration

In [ ]:
# dispensed_database_name = 'app62979_20250203122350'  # Legacy ABM project
dispensed_database_name = 'app879030_20250811132217'  # Update ABM project

output_db_name = 'prescriptions_db'
prescriptions_tb_name = 'dispensed_prescriptions.ht'

prescriptions_tb_sample_001 = 'dispensed_prescriptions_smp_001.ht'
prescriptions_tb_sample_010 = 'dispensed_prescriptions_smp_010.ht'

preferred_partitioning = 32

### Connecting to dispensed database

In [ ]:
dispensed_database_name = dxpy.find_one_data_object(
    classname="database", name=dispensed_database_name, project=dxpy.PROJECT_CONTEXT_ID, describe=True
)["describe"]["name"]
dispensed_database_name

In [ ]:
spark.sql("USE " + dispensed_database_name)

### Fetching data from Spark HQL database

In [ ]:
spark_prescriptions_data = spark.sql(
    """
    SELECT
        eid,
        data_provider AS provider,
        issue_date AS date,
        read_2 AS read2_code,
        bnf_code,
        dmd_code,
        drug_name,
        quantity
    FROM gp_scripts
    """
)

### Load extracted data to Hail table

In [ ]:
spark_prescriptions_data = spark_prescriptions_data.withColumn('date', spark_prescriptions_data.date.cast('string'))

In [ ]:
%%time
hl_prescriptions_data = (
    hl.Table.from_spark(spark_prescriptions_data)
    .repartition(preferred_partitioning)
    .add_index(name = 'idx')
    .select('idx', 'eid', 'provider', 'date', 'read2_code', 'bnf_code', 'dmd_code', 'drug_name', 'quantity')
    .cache()
)

In [ ]:
hl_prescriptions_data.count()

#### Checking integrity of fetched data

In [ ]:
spark_cnt = spark.sql("SELECT COUNT(1) FROM gp_scripts").first()[0]
assert hl_prescriptions_data.count() == spark_cnt

### Write Hail table to dnax database

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_db_name} LOCATION 'dnax://'")

In [ ]:
output_db_id = dxpy.find_one_data_object(name = output_db_name, classname = "database", project = dxpy.PROJECT_CONTEXT_ID)['id']
prescriptions_tb_url = f"dnax://{output_db_id}/{prescriptions_tb_name}"

In [ ]:
%time hl_prescriptions_data.write(prescriptions_tb_url, overwrite = True)

### Write testing samples (downsample to 1% and 10%)

In [ ]:
%time hl_prescriptions_data.sample(0.01).write(f"dnax://{output_db_id}/{prescriptions_tb_sample_001}", overwrite = True)

In [ ]:
%time hl_prescriptions_data.sample(0.10).write(f"dnax://{output_db_id}/{prescriptions_tb_sample_010}", overwrite = True)